# HW 1: Discrete Choice Modeling

Analyze the data contained in the file “YogurtLong.dta.”

First use Stata (or another package) to estimate a multinomial logit brand choice model for this data.  Restrict all coefficients to be identical across choices, except for the intercept for each brand.  Then replicate the results using a pure matrix algebra coding in Matlab, R or comparable language. 

To tie in to future assignments, I suggest creating a master program that calls fminunc.m for maximization.  fminunc will maximize another function which will merely sum the log likelihoods over all N individuals. 

The other function (say Li.m) will create an Nx1 vector of log likelihoods.  Li.m should involve creating an NT by 1 vector of likelihoods for each observation.  Then, use a command such as grpstats.m to take the products within each i to get an N by 1 vector of individual likelihoods that will be output.  (This step is not necessary for this assignment but will be needed in future assignments which is why we are doing it here)

Once this is complete, try adding a right hand side variable to your brand utilities that is an indicator for whether that brand was the last brand purchased.  How do you interpret the results with this added variable?

Also, please come to class prepared to discuss a paper of your choice and the role the data and application plays in helping the paper make a substantive contribution.  If you prefer to be directed to a paper, consider one of the following: 
- Rust (1987)
- Bronnenberg, Dhar and Dube (2009)
- Hartmann and Klapper (2018)

## Results from STATA

```text
Conditional logit choice model                 Number of obs      =     63,920
Case ID variable: choice_id                    Number of cases    =      12784

Alternatives variable: brand                   Alts per case: min =          5
                                                              avg =        5.0
                                                              max =          5

                                                  Wald chi2(2)    =     650.74
Log likelihood = -8781.5944                       Prob > chi2     =     0.0000

------------------------------------------------------------------------------
           b | Coefficient  Std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
brand        |
           p |   -32.7738   1.379016   -23.77   0.000    -35.47662   -30.07097
           f |    .256184   .0876735     2.92   0.003     .0843471    .4280208
-------------+----------------------------------------------------------------
0            |  (base alternative)
-------------+----------------------------------------------------------------
1            |
       _cons |   .9346668   .1472778     6.35   0.000     .6460076    1.223326
-------------+----------------------------------------------------------------
2            |
       _cons |   .3404846   .1183367     2.88   0.004      .108549    .5724202
-------------+----------------------------------------------------------------
3            |
       _cons |  -3.322132   .1386069   -23.97   0.000    -3.593796   -3.050467
-------------+----------------------------------------------------------------
4            |
       _cons |  -.3185887   .1185656    -2.69   0.007     -.550973   -.0862043
------------------------------------------------------------------------------
```

## Python implementation

In [1]:
# Imports
import numpy as np
import pandas as pd
import cvxpy as cp
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess

In [2]:
# Import data
df = pd.read_csv("../data/YogurtLong.txt", sep="\t")
print(df.head())

   hh  qty        exp  nopurch  b1  b2  b3  b4    p1    p2    p3    p4  f1  \
0   1    2  40.900002        0   0   0   0   1  0.11  0.08  0.06  0.08   0   
1   1    0   8.830000        1   0   0   0   0  0.11  0.09  0.05  0.08   0   
2   1    0   3.880000        1   0   0   0   0  0.13  0.09  0.05  0.08   0   
3   1    0   0.780000        1   0   0   0   0  0.13  0.09  0.05  0.08   0   
4   1    0  39.240002        1   0   0   0   0  0.13  0.09  0.05  0.08   0   

   f2  f3  f4  tripnum  
0   0   0   0        1  
1   0   0   0        2  
2   0   0   0        3  
3   0   0   0        4  
4   0   0   0        5  


In [3]:
# Convert df to long (each row is a hh-trip-brand choice occasion)
df_long = (
    pd.wide_to_long(
        df,
        stubnames=["b", "p", "f"],
        i=["hh", "tripnum"],
        j="brand",
        suffix=r"\d+"
    )
    .reset_index()
    .sort_values(["hh", "tripnum", "brand"])
)
df_long = pd.get_dummies(df_long, columns=["brand"], prefix="brand", dtype=int)
df_long.head()

,hh,tripnum,exp,nopurch,qty,b,p,f,brand_1,brand_2,brand_3,brand_4
0,1,1,40.900002,0,2,0,0.11,0,1,0,0,0
1,1,1,40.900002,0,2,0,0.08,0,0,1,0,0
2,1,1,40.900002,0,2,0,0.06,0,0,0,1,0
3,1,1,40.900002,0,2,1,0.08,0,0,0,0,1
4,1,2,8.830000,1,0,0,0.11,0,1,0,0,0


### I. `cvxpy` implementation

Note that the log-likelihood is concave. Let $i = 1, \ldots H$ index households and $t = 1, \ldots, T_h$ index household $i$'s trips. Then, the log-likelihood of observing household $i$ choose brand $j$ in trip $t$ is
\begin{align*}
\log\left( \prod_{i=1}^H \prod_{t=1}^{T_h} \Pr(y = j \mid x\beta) \right) 
&= \log\left( \prod_{i=1}^H \prod_{t=1}^{T_h} \frac{\exp(\tilde{v}_{ij})}{1 + \sum_{k=1}^J \exp(\tilde{v}_{ik})} \right) \\
&= \sum_{i=1}^H \sum_{t=1}^{T_h} \left(\tilde{v}_{ij} - \log\left(1 + \sum_{k=1}^J \exp(\tilde{v}_{ik})\right) \right) \\
&= \sum_{i=1}^H \sum_{t=1}^{T_h} \left(\tilde{v}_{ij} - \log\left(\exp(0) + \sum_{k=1}^J \exp(\tilde{v}_{ik})\right) \right)
\end{align*}
As the log-sum-exp function is convex (Boyd and Vandenberghe) and $v_{ik}$ is an affine function of $\beta$, the summand is concave in $\beta$ and so the log-likehihood function is concave. We can proceed by using a convex optimization solver (i.e., `cvxpy`) to maximize a concave function.

In [4]:
# Form X, Y matrices
coef_names = ['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p']
X = df_long[coef_names].to_numpy()
Y = df_long[['b']].to_numpy()

# Compute v_ij's
n_alts = 4
n, k = X.shape
beta = cp.Variable((k, 1))
v_ij = X @ beta

V = cp.reshape(v_ij, (n // n_alts, n_alts), order='C')
V = cp.hstack([np.zeros((n // n_alts, 1)), V])  # add zero utility for outside option
chosen_v = Y.T @ v_ij

loglik = chosen_v - cp.sum(cp.log_sum_exp(V, axis=1))
problem = cp.Problem(cp.Maximize(loglik))
opt_ll = problem.solve()
opt_beta = beta.value.flatten()

In [5]:
# Print coefficient estimates
for name, coef in zip(coef_names, opt_beta):
    print(f"{name:<10} | {coef:>12.4f}")

brand_1    |       0.9347
brand_2    |       0.3405
brand_3    |      -3.3221
brand_4    |      -0.3186
f          |       0.2562
p          |     -32.7738


### II. Generic optimizer implementation

I suppose the log-likelihood might not always be concave for more complicated models, so it is probably best to implement a solver that doesn't rely on convexity of the objective.

In [ ]:
def loglik(beta, X, Y, hh_ids, n_alts):
    """
    Computes negative log-likelihood for the dataset, assuming a conditional logit model with an outside option.

    Parameters
    ----------
    beta : array
        Coefficients to be estimated.
    X : array
        Matrix of covariates
    Y : array
        Vector of choices
    hh_ids : array
        Household ID corresponding to each row of X and Y.
    n_alts : int
        Number of alternatives.

    Returns
    -------
    float
        Negative log-likelihood.
    """
    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)
    hh_ids = np.asarray(hh_ids).reshape(-1)
    beta = np.asarray(beta).reshape(-1)

    n, k = X.shape
    n_choices = n // n_alts                         # number of hh-trip choice occasions

    v_ij = X @ beta                                 # utility for each hh-trip-brand
    V = v_ij.reshape(n_choices, n_alts, order="C")  # reshape to (choice occasion, alternative)
    chosen_v = np.multiply(Y, v_ij).\
        reshape(n_choices, n_alts).sum(axis=1)      # chosen utility per choice occasion
    V_with_outside = np.hstack([np.zeros((n_choices, 1)), V])   # add outside option (zero utility)
    log_denom = logsumexp(V_with_outside, axis=1)

    loglik_trip = chosen_v - log_denom                          # trip-level log-likelihoods
    _, hh_index = np.unique(hh_ids, return_inverse=True)        # group by household
    loglik_hh = np.bincount(hh_index, weights=loglik_trip)

    return -np.sum(loglik_hh)


def score_matrix(beta, X, Y, n_alts):
    """
    Returns choice-occasion score vectors s_t(beta) = d log L_t / d beta.
    Trip-level aggregation (so this isn't cluster-robust to household correlation).
    Shape: (n_choices, k).
    """
    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)
    beta = np.asarray(beta).reshape(-1)

    n, k = X.shape
    n_choices = n // n_alts

    # Reshape long format into (choice occasion, alternative, regressor)
    X3 = X.reshape(n_choices, n_alts, k, order="C")
    Y2 = Y.reshape(n_choices, n_alts, order="C")

    # Softmax probabilities including outside option with normalized utility 0
    V = X3 @ beta                                      # (n_choices, n_alts)
    V_with_outside = np.hstack([np.zeros((n_choices, 1)), V])
    P_with_outside = np.exp(V_with_outside - logsumexp(V_with_outside, axis=1, keepdims=True))
    P = P_with_outside[:, 1:]                          # drop outside option prob

    # Score per choice occasion: sum_j (y_tj - p_tj) x_tj
    S = np.einsum("ta,tak->tk", (Y2 - P), X3)
    return S


def mle_estimator(X, Y, beta_init, hh_ids, n_alts,
                  ci_alpha=0.05,
                  robust_se=False,
                  method='BFGS'):
    """
    Estimates the multinomial logit model using maximum likelihood estimation.
    
    Parameters    
    ----------
    X : array
        Matrix of covariates.
    Y : array
        Vector of choices.
    beta_init : array
        Initial guess for the coefficients.
    hh_ids : array
        Household ID corresponding to each row of X and Y.
    n_alts : int
        Number of alternatives.
    ci_alpha : float
        Significance level for confidence intervals.
    robust_se : bool
        Whether to compute robust standard errors.
    method : str
        Optimization method.

    Returns    
    -------
    Dictionary containing:
        - optimized log-likelihood 
        - coefficient estimates
        - standard errors
        - confidence intervals for coefficients
    """    
    result = minimize(
        loglik, 
        beta_init, 
        args=(X, Y, hh_ids, n_alts),
        method=method,
    )

    opt_ll = -result.fun
    opt_beta = result.x

    # Standard errors and confidence intervals (using Hessian, under correct specification)
    H = approx_hess(opt_beta, loglik, args=(X, Y, hh_ids, n_alts))  # numerical Hessian at opt_beta
    H_inv = np.linalg.inv(H)
    se_beta = np.sqrt(np.diag(H_inv))
    z_score = norm.ppf(1 - ci_alpha / 2)
    ci = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_beta, se_beta)]

    output = {
        'opt_ll': opt_ll,
        'opt_beta': opt_beta,
        'se_beta': se_beta,
        'ci': ci,
    }

    if robust_se:
        # J = E[s_t s_t'] estimated by sample average of outer products of scores.
        S = score_matrix(opt_beta, X, Y, n_alts)       # (n_choices, k)
        n_choices = S.shape[0]
        J = S.T @ S

        # Sandwich variance: H^{-1} J H^{-1}
        V_robust = H_inv @ J @ H_inv
        se_beta_r = np.sqrt(np.diag(V_robust))
        ci_r = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_beta, se_beta_r)]

        output['se_beta'] = se_beta_r
        output['ci'] = ci_r

    return output

In [8]:
# Form X, Y matrices
coef_names = ['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p']
hh_ids = df[['hh']].to_numpy()
X = df_long[coef_names].to_numpy()
Y = df_long[['b']].to_numpy()

n_alts = 4
n, k = X.shape
beta_init = np.zeros(k)  # initial guess for beta

results = mle_estimator(X, Y, beta_init, hh_ids, n_alts, robust_se=False)
opt_ll = results['opt_ll']
opt_beta = results['opt_beta']
se_beta = results['se_beta']

# Print coefficient estimates
print(f"{'Coefficient':<10} | {'Estimate':>10} | {'Std. Err.':>10} | {'95% CI':>10}")
print("-" * 59)
for i, (name, coef) in enumerate(zip(coef_names, opt_beta)):
    print(f"{name:<11} | {coef:>10.4f} | {se_beta[i]:>10.4f} | {results['ci'][i][0]:>10.3f}, {results['ci'][i][1]:>7.3f}")

AttributeError: 'numpy._ArrayFunctionDispatcher' object has no attribute 'ppf'

## Adding brand inertia

In [10]:
df_long["prev_purch"] = df_long.groupby("hh")["b"].shift(4).fillna(0).astype(int)
df_long.head()

# Form X, Y matrices
coef_names = ['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch']
hh_ids = df[['hh']].to_numpy()
X = df_long[coef_names].to_numpy()
Y = df_long[['b']].to_numpy()

# Compute v_ij's
n_alts = 4
n, k = X.shape
beta_init = np.zeros(k)  # initial guess for beta

result = minimize(
    loglik, 
    beta_init, 
    args=(X, Y, hh_ids, n_alts),
    method='BFGS',  # quasi-Newton
)

opt_ll = -result.fun
opt_beta = result.x

Coefficient  |  Estimate  | Std. Err.  |   Confidence Interval  
-------------+------------+------------+------------+-----------
brand_1      |   0.562170 |  0.3194084 |   -0.06386 |    1.18820
brand_2      |  -0.144805 |  0.2966589 |   -0.72625 |    0.43664
brand_3      |  -3.361725 |  0.2873736 |   -3.92497 |   -2.79848
brand_4      |  -0.632828 |  0.2760786 |   -1.17393 |   -0.09172
f            |   0.353774 |  0.1230793 |    0.11254 |    0.59500
p            | -35.140550 |  3.3119919 |  -41.63193 |  -28.64916
prev_purch   |   3.182005 |  0.1102992 |    2.96582 |    3.39819


In [ ]:
# Print coefficient estimates
for name, coef in zip(coef_names, opt_beta):
    print(f"{name:<10} | {coef:>12.4f}")

brand_1    |       0.5622
brand_2    |      -0.1448
brand_3    |      -3.3617
brand_4    |      -0.6328
f          |       0.3538
p          |     -35.1405
prev_purch |       3.1820


The coefficient on `prev_purch` is 3.1820 and statistically significant, suggesting that previously purchasing a brand is associated with a 3.182 increase in the log-odds ($\exp(3.182) = 24.095$ in the odds ratio) of re-purchasing that brand (relative to the outside option) in the next trip. One could take this result to suggest that there is brand inertia.